# Agentic Windows UI Automation Research

To achieve true remote control over your native Windows applications (like Unreal Engine) without relying on brittle screen-scraping or coordinate guessing, we must tap into the underlying accessibility layers of the OS. 

Here is a deep dive into the available native solutions, existing frameworks, and the architectural hurdle we must cross to make it work.

## 1. The Core Technology: Microsoft UI Automation (UIA)
Modern "Computer-Using Agents" (CUAs) bypass screenshots entirely by integrating with the **Microsoft UI Automation (UIA)** API. 
*   **Structured UI Trees:** UIA is the native Windows API used by screen readers. It exposes every running application as a structured, hierarchical tree of elements (Buttons, TextBoxes, Menus) complete with their properties, names, and interaction patterns.
*   **Deterministic Actions:** Instead of calculating X/Y coordinates to click a pixel, an agent queries the UIA tree for a button by its AutomationId or Name, and triggers its `InvokePattern`. This works reliably even if the window is moved, resized, or partially obscured.

## 2. Existing MCP Server Implementations
Instead of building a UIA wrapper from scratch in C# or Python, the open-source community has recently published several robust Model Context Protocol (MCP) servers that expose the UIA API directly to agents:

1.  **WinApp MCP / Windows-MCP:** Highly popular node-based servers that bridge LLMs and Windows UIA. They provide explicit tools for GUI automation (`inspect_ui`, `click_element`, `get_window_state`), file navigation, and application launching. 
2.  **UIAutomation MCP:** A specialized, high-performance server built with native AOT (Ahead-of-Time) compilation. It emphasizes low memory usage and comprehensive access to all UIA Control Patterns.

## 3. The Architectural Hurdle: Session 0 Isolation
As we discovered when my `AppActivate` keystroke script failed earlier, my agent process is running as a background service. In Windows architecture, background services run in **Session 0**, which is strictly isolated from the interactive user desktop (**Session 1/2**). 

Even if I install and spawn a native UIA MCP server using my terminal execution tools, that server will inherit my Session 0 restrictions and will **not** be able to see or interact with your foreground Unreal Engine window. 

## 4. The Implementation Plan
We do not need to build the automation layer from scratch, but we *do* need to bridge the session gap. Here is how we achieve full native control:

1.  **Select the Server:** We will use a community UIA MCP server (e.g., `WinApp MCP`).
2.  **User-Session Execution:** Instead of configuring Antigravity to spawn the server automatically (which traps it in Session 0), you will run the MCP server in your standard foreground PowerShell terminal using `npx` or as a standalone executable.
3.  **SSE Transport Integration:** We will configure our `mcp_config.json` to connect to that server via HTTP/Server-Sent Events (SSE) on localhost.
4.  **Agentic Control:** Once connected, I will dynamically inherit tools allowing me to query the live UIA tree of your desktop and send deterministic clicks and keystrokes directly into Unreal Engine.

**Conclusion:** The technology exists natively via UIA, and the MCP wrappers are already built. We just need to deploy the server in your interactive desktop session and connect to it over local SSE.